# LogSeer Prediction

Predicts whether a planned operation will succeed or fail based on current server logs.

**Setup**: copy your data to `DATA_DIR`. Two layouts are supported:

```
# Single prediction — log files directly in DATA_DIR:
DATA_DIR/
├── log1.txt
└── log2.txt

# Multiple predictions — one subdirectory per operation run:
DATA_DIR/
├── 0001/
│   ├── log1.txt
│   └── log2.txt
└── 0002/
    └── log1.txt
```

If you already ran training on this machine, model files are already in the repo root — only `DATA_DIR` needs to be set up. Otherwise copy `NN_MODEL_FILE`, `TOKENIZER_FILE`, and `SK_MODEL_FILE` to the repo root, or use the Colab cell below to load them from Google Drive.

**Output**: each row in the results is one operation run, with `OR` and `AND` ensemble predictions — use `AND` as the high-precision signal.

In [ ]:
# Setup
import os, sys

# On Google Colab: clone the repo and set up the path
# On local: assumes notebook is run from the notebooks/ directory

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if not os.path.exists('logseer'):
    os.system('git clone https://github.com/masahiko-shibata/logseer.git')
    os.chdir('logseer')

sys.path.insert(0, '.')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from logseer import Seer, print_results

In [ ]:
# Configuration
MODEL_BASE_DIR            = './'                           # directory where trained models are stored
DATA_DIR            = 'data_current'
NN_MODEL_FILE       = 'logseer.keras'
TOKENIZER_FILE      = 'tokenizer.pickle'
SK_MODEL_FILE       = 'xgb.pkl'
NN_MODEL_PATH       = MODEL_BASE_DIR + NN_MODEL_FILE
TOKENIZER_PATH      = MODEL_BASE_DIR + TOKENIZER_FILE
SK_MODEL_PATH       = MODEL_BASE_DIR + SK_MODEL_FILE
NUMCHAR             = 3000
MAX_SEQUENCE_LENGTH = 26000
NN_THRESHOLD        = 0.72
XGB_THRESHOLD       = 0.52

**Colab only** — Run below to copy data and trained models from Google Drive. Otherwise copy them manually and skip this cell.

In [ ]:
# Copy data and trained models from Google Drive.
from google.colab import drive
import shutil, zipfile
drive.mount('/content/drive')

DRIVE_BASE      = '/content/drive/MyDrive/Colab Notebooks/logseer/'
DRIVE_DATA_DIR  = DRIVE_BASE + '/data/'
DRIVE_MODEL_DIR = DRIVE_BASE + '/models/'
DATA_ZIP        = 'data_current.zip'

shutil.copy(DRIVE_MODEL_DIR + NN_MODEL_FILE,  NN_MODEL_PATH)
shutil.copy(DRIVE_MODEL_DIR + TOKENIZER_FILE, TOKENIZER_PATH)
shutil.copy(DRIVE_MODEL_DIR + SK_MODEL_FILE,  SK_MODEL_FILE)
shutil.copy(DRIVE_DATA_DIR + DATA_ZIP,  DATA_ZIP)
with zipfile.ZipFile(DATA_ZIP, 'r') as z:
    z.extractall()
print(os.listdir())

Run the cell below to load models and predict.

In [ ]:
# Load models and predict
seer    = Seer.from_files(NN_MODEL_PATH, TOKENIZER_PATH, SK_MODEL_FILE,
                          numchar=NUMCHAR, max_sequence_length=MAX_SEQUENCE_LENGTH,
                          nn_threshold=NN_THRESHOLD, xgb_threshold=XGB_THRESHOLD)
results = seer.predict(DATA_DIR)
print_results(results)